# oc-direct — run it on Colab or Kaggle

OpenAI-compatible gateway for the OpenCode free tier. This notebook is end-to-end:
download → install Bun → configure → start → test.

Only prerequisite: outbound internet (**Kaggle: Settings → Internet → On**).

Re-running a cell is safe. The only upstream quota spent is the single completion in
step 6.


## 1. Download and stage the release


In [1]:
import os, shutil, subprocess, urllib.request, zipfile

ZIP_URL = "https://github.com/Subham460/ai-gateway/raw/refs/heads/main/oc-direct.zip"
ZIP     = "/tmp/oc-direct.zip"
WORK    = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
DEST    = os.path.join(WORK, "oc-direct")

if not os.path.exists(ZIP) or os.path.getsize(ZIP) < 10_000:
    urllib.request.urlretrieve(ZIP_URL, ZIP)
print("zip:", os.path.getsize(ZIP), "bytes")

STAGE = "/tmp/ocd-stage"
shutil.rmtree(STAGE, ignore_errors=True)
os.makedirs(STAGE, exist_ok=True)
if shutil.which("unzip"):
    subprocess.run(["unzip", "-q", "-o", ZIP, "-d", STAGE], check=True)
else:
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(STAGE)

root = STAGE if os.path.exists(f"{STAGE}/package.json") else f"{STAGE}/oc-direct"
assert os.path.exists(f"{root}/package.json"), f"unexpected layout: {os.listdir(STAGE)}"

os.makedirs(DEST, exist_ok=True)
for name in os.listdir(root):
    src, dst = f"{root}/{name}", f"{DEST}/{name}"
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)
os.chdir(DEST)
print("dest:", DEST, "| entries:", len(os.listdir(DEST)))
print("prompt bytes:", os.path.getsize("src/assets/system-prompt.txt"), "(expect 8605)")


zip: 98701 bytes
dest: /content/oc-direct | entries: 20
prompt bytes: 8605 (expect 8605)


## 2. Install Bun (no-op if present)


In [2]:
import os, subprocess

BUN = os.path.expanduser("~/.bun/bin/bun")
if not os.path.exists(BUN):
    subprocess.run("curl -fsSL https://bun.sh/install | bash", shell=True, check=True)
os.environ["PATH"] = os.path.dirname(BUN) + ":" + os.environ["PATH"]
print("bun", subprocess.run(["bun", "--version"], capture_output=True, text=True).stdout.strip())


bun 1.4.4


## 3. Generate the gateway key and `.env`

Loopback bind + API key: nothing is exposed, and clients authenticate normally.


In [3]:
import os

os.chdir(DEST)

KEY = "35893d2f57cdeef78a8aa7142ffb7db6240fe10d66130261"

with open(".env", "w") as f:
    f.write(f"""OCD_KEYS={KEY}:notebook
OCD_HOST=127.0.0.1
OCD_PORT=8787
OCD_LOG_LEVEL=info
OCD_TOOLS_MODE=full
OCD_TOOL_CHOICE=auto
""")

os.chmod(".env", 0o600)

with open(".env.key", "w") as f:
    f.write(KEY)

os.chmod(".env.key", 0o600)

print("Gateway key:", KEY)

Gateway key: 35893d2f57cdeef78a8aa7142ffb7db6240fe10d66130261


## 4. Start the gateway


In [4]:
import os, subprocess, time

os.chdir(DEST)
os.environ["PATH"] = os.path.expanduser("~/.bun/bin") + ":" + os.environ["PATH"]

subprocess.run(["pkill", "-f", "src/server.ts"], capture_output=True)
time.sleep(1)
log = open("/tmp/ocd.log", "w")
proc = subprocess.Popen(["bash", "run.sh"], cwd=DEST,
                        stdout=log, stderr=subprocess.STDOUT, start_new_session=True)

code = "000"
for _ in range(30):
    time.sleep(1)
    code = subprocess.run(["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}",
                           "http://127.0.0.1:8787/healthz"], capture_output=True, text=True).stdout
    if code == "200":
        break
print("healthz:", code, "| pid:", proc.pid)
print(open("/tmp/ocd.log").readline()[:400])


healthz: 200 | pid: 890
{"ts":"2026-11-16T11:13:55.129Z","level":"info","msg":"listening","service":"oc-direct","url":"http://127.0.0.1:8787","port":8787,"models":"deepseek-v4.1-flash-free,flint-40b-free,glm-5.3-flash-free,big-pickle,iceman-free,inkling-2-small-free,longcat-2.5-free,mimo-v2.6-flash-free,minimax-m3.1-flash-free,nemotron-3.5-lightning-free,nemotron-3.5-super-vl-free,beam-free","toolsMode":"full","maxConcur


## 5. Unit tests — offline, zero quota


In [5]:
import subprocess
print(subprocess.run(["bun", "test"], cwd=DEST, capture_output=True, text=True).stdout[-500:])


bun test v1.4.4 (38afee6c8)



## 6. End-to-end smoke test — one real completion


In [6]:
import subprocess
print(subprocess.run(["bash", "scripts/smoke.sh"], cwd=DEST,
                     capture_output=True, text=True).stdout[-900:])


[smoke] healthz 200
[smoke] unauthenticated request rejected (401)
[smoke] models listed
[smoke] readyz ok
[smoke] completion ok: ok
{"id":"chatcmpl-e6428200a51047908cb7132881e5e3fa","object":"chat.completion","created":1794827637,"model":"deepseek-v4.1-flash-free","choices":[{"index":0,"message":{"role":"assistant","content":"ok"},"finish_reason":"stop"}],"usage":{"prompt_tokens":5346,"completion_tokens":3,"total_tokens":5349,"prompt_tokens_details":{"cached_tokens":1920},"completion_tokens_details":{"reasoning_tokens":0}}}

[smoke] all checks passed



## 7. Streaming from Python


In [7]:
import json, requests

KEY = open(f"{DEST}/.env.key").read().strip()
r = requests.post(
    "http://127.0.0.1:8787/v1/chat/completions",
    headers={"authorization": f"Bearer {KEY}"},
    json={"model": "deepseek-v4.1-flash-free",
          "messages": [{"role": "user", "content": "count to five"}],
          "stream": True},
    stream=True, timeout=180,
)
r.raise_for_status()
for line in r.iter_lines(decode_unicode=True):
    if not line or not line.startswith("data: "):
        continue
    data = line[6:]
    if data == "[DONE]":
        break
    delta = json.loads(data)["choices"][0]["delta"]
    print(delta.get("reasoning_content") or delta.get("content") or "", end="")


simple request: count to five.1, 2, 3, 4, 5.

## 8. Optional — official OpenAI SDK


In [8]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai"], check=True)

from openai import OpenAI
client = OpenAI(base_url="http://127.0.0.1:8787/v1",
                api_key=open(f"{DEST}/.env.key").read().strip())
r = client.chat.completions.create(
    model="deepseek-v4.1-flash-free",
    messages=[{"role": "user", "content": "one sentence about tide pools"}],
)
print(r.choices[0].message.content)
print(r.usage)


Tide pools are small pools of seawater left behind in rocky depressions when the tide recedes, forming isolated microhabitats where hardy creatures like anemones, crabs, and starfish shelter until the water returns.
CompletionUsage(completion_tokens=56, prompt_tokens=5344, total_tokens=5400, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=11, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cache_write_tokens=None, cached_tokens=1920))


## 9. Optional — contract check (~6 live calls)

Confirms the provider's attested request shape has not moved.


In [9]:
import subprocess
print(subprocess.run(["bun", "run", "verify:contract"], cwd=DEST,
                     capture_output=True, text=True).stdout[-800:])


OK    canonical request                          served    (200 data: {"id":"gw-2b6defcc5f88-6ba295","object":"chat.completion.chunk","created":) 1287ms
OK    system prompt replaced by a one-liner      rejected  (429 FreeUsageLimitError) 520ms
OK    no tools                                   rejected  (429 FreeUsageLimitError) 270ms
OK    only a non-read tool                       rejected  (429 FreeUsageLimitError) 253ms
OK    stream disabled                            rejected  (429 FreeUsageLimitError) 272ms
OK    random session id                          rejected  (429 FreeUsageLimitError) 255ms

contract unchanged: all probes match the recorded wire requirements



In [10]:
print(KEY)

35893d2f57cdeef78a8aa7142ffb7db6240fe10d66130261


In [11]:
import os, platform, urllib.request

arch = {
    "x86_64": "amd64",
    "aarch64": "arm64",
}[platform.machine()]

url = f"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}"

CLOUDFLARED = "/tmp/cloudflared"
urllib.request.urlretrieve(url, CLOUDFLARED)
os.chmod(CLOUDFLARED, 0o755)

print("Installed:", os.popen(f"{CLOUDFLARED} --version").read().strip())

Installed: cloudflared version 2026.11.0 (built 2026-11-12-12:17 UTC)


## Troubleshooting

| Symptom | Fix |
| --- | --- |
| HTTPError 404 on download | zip not pushed to that branch yet, or repo is private |
| unexpected layout | re-run step 1; archive must contain oc-direct/ or package.json |
| bun: command not found | re-run step 2 - PATH is per kernel |
| Could not resolve host | Kaggle internet toggle off / no Colab network |
| healthz: 000 | server died; see /tmp/ocd.log |
| 401 from smoke test | delete .env and .env.key, re-run step 3 |
| 429 FreeUsageLimitError | upstream throttle; re-run shortly or run step 9 |
| empty content, finish_reason length | reasoning consumed budget - raise max_tokens |

**Platform notes.** Colab wipes /content, ~/.bun, and /tmp on recycle. Kaggle keeps
/kaggle/working only with *Save Version*, has no inbound ports, and caps sessions at
~12 h. Both are demo runners; for a host use deploy/docker-compose.yml (see
PRODUCTION.md).


In [12]:
!curl -fsSL https://opencode.ai/v2/install | bash


Installing opencode version: 2.3.0
■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■ 100%
Successfully added opencode to $PATH in /root/.bashrc

                                 ▄     
█▀▀█ █▀▀█ █▀▀█ █▀▀▄ █▀▀▀ █▀▀█ █▀▀█ █▀▀█
█░░█ █░░█ █▀▀▀ █░░█ █░░░ █░░█ █░░█ █▀▀▀
▀▀▀▀ █▀▀▀ ▀▀▀▀ ▀  ▀ ▀▀▀▀ ▀▀▀▀ ▀▀▀▀ ▀▀▀▀


OpenCode includes free models, to start:

exec $SHELL   # Reload PATH (or open a new terminal)
cd <project>  # Open directory
opencode      # Run command

For more information visit https://opencode.ai/docs




In [13]:
import os
import re
import time
import requests
import subprocess
from google.colab import userdata

# Bifrost deployment
BIFROST_URL = "https://subhamgiri9748-bifrost.hf.space".rstrip("/")
PROVIDER = "colab"

# Gateway key generated for OCD
GATEWAY_KEY = KEY

# Existing cloudflared binary
# CLOUDFLARED must already be defined

# Provider key name in Bifrost
KEY_NAME = "temp"

# Cloudflare tunnel
LOG = "/tmp/cloudflared.log"

print("Configuration loaded")
print("Bifrost:", BIFROST_URL)
print("Provider:", PROVIDER)
print("Gateway key loaded:", bool(GATEWAY_KEY))
print("Cloudflared:", CLOUDFLARED)


Configuration loaded
Bifrost: https://subhamgiri9748-bifrost.hf.space
Provider: colab
Gateway key loaded: True
Cloudflared: /tmp/cloudflared


In [14]:
username = userdata.get("BIFROST_ADMIN_USERNAME")
password = userdata.get("BIFROST_ADMIN_PASSWORD")

session = requests.Session()

response = session.post(
    f"{BIFROST_URL}/api/session/login",
    json={
        "username": username,
        "password": password
    },
    timeout=30
)

print("Login status:", response.status_code)
print("Cookies:", list(session.cookies.keys()))

response.raise_for_status()

print("Bifrost authentication successful")


Login status: 200
Cookies: ['token']
Bifrost authentication successful


In [15]:
# Get provider configuration
response = session.get(
    f"{BIFROST_URL}/api/providers/{PROVIDER}",
    timeout=30
)

print("Provider status:", response.status_code)
response.raise_for_status()

provider_config = response.json()

print("\nProvider configuration:")
print("Name:", provider_config.get("name"))
print("Network:", provider_config.get("network_config"))
print("Custom config:", provider_config.get("custom_provider_config"))

# Get separately managed provider keys
response = session.get(
    f"{BIFROST_URL}/api/providers/{PROVIDER}/keys",
    timeout=30
)

print("\nKeys status:", response.status_code)
response.raise_for_status()

keys_data = response.json()
provider_keys = keys_data.get("keys", [])

print("\nConfigured keys:")

for item in provider_keys:
    print({
        "id": item.get("id"),
        "name": item.get("name"),
        "enabled": item.get("enabled"),
        "models": item.get("models")
    })


Provider status: 200

Provider configuration:
Name: colab
Network: {'base_url': 'https://eliminate-rare-stevens-meyer.trycloudflare.com', 'default_request_timeout_in_seconds': 300, 'max_retries': 0, 'retry_backoff_initial': 500, 'retry_backoff_max': 5000, 'stream_idle_timeout_in_seconds': 120, 'keep_alive_timeout_in_seconds': 30, 'max_conns_per_host': 5000}
Custom config: {'is_key_less': False, 'base_provider_type': 'openai', 'allowed_requests': {'list_models': True, 'model_retrieve': True, 'text_completion': True, 'text_completion_stream': True, 'chat_completion': True, 'chat_completion_stream': True, 'responses': True, 'responses_stream': True, 'responses_retrieve': True, 'responses_delete': True, 'responses_cancel': True, 'responses_input_items': True, 'count_tokens': True, 'compaction': False, 'embedding': True, 'rerank': True, 'decisions': True, 'ocr': True, 'speech': True, 'speech_stream': True, 'transcription': True, 'transcription_stream': True, 'image_generation': True, 'image

In [16]:

# Stop previous tunnel if this cell is rerun
if "tunnel" in globals() and tunnel.poll() is None:
    tunnel.terminate()
    tunnel.wait(timeout=10)

log_file = open(LOG, "w")

tunnel = subprocess.Popen(
    [
        CLOUDFLARED,
        "tunnel",
        "--no-autoupdate",
        "--url",
        "http://127.0.0.1:8787"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

TUNNEL_URL = None
output = ""

for attempt in range(60):
    if tunnel.poll() is not None:
        raise RuntimeError("Cloudflared exited unexpectedly")

    time.sleep(1)

    with open(LOG) as f:
        output = f.read()

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        output
    )

    if match:
        TUNNEL_URL = match.group(0)
        break

if not TUNNEL_URL:
    raise RuntimeError("Tunnel URL not found:\n" + output)

NEW_ENDPOINT = TUNNEL_URL

print("Tunnel established")
print("Public URL:", TUNNEL_URL)
print("API endpoint:", NEW_ENDPOINT)


Tunnel established
Public URL: https://picks-controversy-mention-anytime.trycloudflare.com
API endpoint: https://picks-controversy-mention-anytime.trycloudflare.com


In [17]:
import copy
import time

time.sleep(40)  # Wait 10 seconds before starting

provider_url = f"{BIFROST_URL}/api/providers/{PROVIDER}"

# Fetch fresh configuration
response = session.get(provider_url, timeout=30)
response.raise_for_status()

config = copy.deepcopy(response.json())

# Remove response-only fields
for field in (
    "name",
    "keys",
    "provider_status",
    "status",
    "description",
    "config_hash"
):
    config.pop(field, None)

# Preserve existing network settings
network_config = config.get("network_config") or {}
network_config["base_url"] = NEW_ENDPOINT

config["network_config"] = network_config

# Update provider
response = session.put(
    provider_url,
    json=config,
    timeout=60
)

print("Update status:", response.status_code)
print("Response:", response.text[:1500])

response.raise_for_status()

print("Provider endpoint update successful")


Update status: 200
Response: {"name":"colab","network_config":{"base_url":"https://picks-controversy-mention-anytime.trycloudflare.com","default_request_timeout_in_seconds":300,"max_retries":0,"retry_backoff_initial":500,"retry_backoff_max":5000,"stream_idle_timeout_in_seconds":120,"keep_alive_timeout_in_seconds":30,"max_conns_per_host":5000},"concurrency_and_buffer_size":{"concurrency":1000,"buffer_size":5000},"proxy_config":null,"send_back_raw_request":false,"send_back_raw_response":false,"store_raw_request_response":false,"ignore_provider_cost":false,"custom_provider_config":{"is_key_less":false,"base_provider_type":"openai","allowed_requests":{"list_models":true,"model_retrieve":true,"text_completion":true,"text_completion_stream":true,"chat_completion":true,"chat_completion_stream":true,"responses":true,"responses_stream":true,"responses_retrieve":true,"responses_delete":true,"responses_cancel":true,"responses_input_items":true,"count_tokens":true,"compaction":false,"embedding":tr

In [18]:

# Fetch current keys again
keys_url = f"{provider_url}/keys"

response = session.get(keys_url, timeout=30)
response.raise_for_status()

provider_keys = response.json().get("keys", [])

# Find the exact key by name
matches = [
    item for item in provider_keys
    if item.get("name") == KEY_NAME
]

if len(matches) != 1:
    raise RuntimeError(
        f"Expected exactly one key named {KEY_NAME!r}, "
        f"found {len(matches)}. Inspect Cell 3."
    )

target_key = matches[0]
KEY_ID = target_key["id"]

print("Updating key:")
print("Name:", target_key.get("name"))
print("ID:", KEY_ID)

# Dedicated provider-key update endpoint.
# Patch semantics preserve fields not included in this payload.
key_update_url = f"{keys_url}/{KEY_ID}"

response = session.put(
    key_update_url,
    json={
        "value": GATEWAY_KEY
    },
    timeout=60
)

print("Key update status:", response.status_code)
print("Response:", response.text[:1500])

response.raise_for_status()

print("Provider API key updated successfully")


Updating key:
Name: temp
ID: 0be48df2-9fd3-4e1a-830f-82831a67ecde
Key update status: 200
Response: {"id":"0be48df2-9fd3-4e1a-830f-82831a67ecde","name":"temp","value":{"value":"3589************************0261","type":"plain_text"},"models":["*"],"blacklisted_models":[],"weight":1,"enabled":true,"use_for_batch_api":false,"use_anthropic_endpoints":false,"use_openai_endpoints":false,"status":"success"}

Provider API key updated successfully


In [19]:

# Verify provider endpoint
response = session.get(provider_url, timeout=30)
response.raise_for_status()

saved_config = response.json()

saved_endpoint = (
    saved_config.get("network_config", {}).get("base_url")
)

# Verify key metadata
response = session.get(keys_url, timeout=30)
response.raise_for_status()

saved_keys = response.json().get("keys", [])

saved_key = next(
    (k for k in saved_keys if k.get("id") == KEY_ID),
    None
)

print("========== VERIFICATION ==========")

print("Endpoint:")
print("Expected:", NEW_ENDPOINT)
print("Actual:  ", saved_endpoint)
print("Match:   ", saved_endpoint == NEW_ENDPOINT)

print("\nAPI key:")
print("Key exists:", saved_key is not None)
print("Key enabled:", saved_key.get("enabled") if saved_key else None)
print("Key name:", saved_key.get("name") if saved_key else None)

if saved_endpoint != NEW_ENDPOINT or saved_key is None:
    raise RuntimeError("Verification failed")

print("\nSUCCESS: Bifrost configuration updated.")


========== VERIFICATION ==========
Endpoint:
Expected: https://picks-controversy-mention-anytime.trycloudflare.com
Actual:   https://picks-controversy-mention-anytime.trycloudflare.com
Match:    True

API key:
Key exists: True
Key enabled: True
Key name: temp

SUCCESS: Bifrost configuration updated.
